# Module 4: Designing a Vector Search System

Qdrant Beginners Course, follow-along notebook.

Course page: https://qdrant.tech/course/beginners/module-4/

## Recap: Modules 1-3

Embeddings + cosine similarity for meaning. Qdrant's data model: collection, point, vector, payload. HNSW for fast search, payload filters for exact matches. Hybrid search combines dense (meaning) and sparse (exact tokens) via `Prefetch` + fusion.

This module: the five layers of a vector search system, and what to decide before you ingest.


In [ ]:
!pip install -q "qdrant-client[fastembed]" 

## Five layers

1. **Query**: embed, search, fuse, return top-K. Fix and rerun anytime.
2. **Indexing**: HNSW graph + payload indexes. A mistake here is slow, not wrong, rebuild fixes it.
3. **Storage**: memory vs disk. Quantization and on-disk vectors trade precision/latency for capacity.
4. **Data**: content, model, chunk size, payload schema. No later layer can fix a data mistake, you have to re-ingest.
5. **Distribution**: sharding and replication once one machine isn't enough.

## Decide before you ingest

Example brief: analysts search global news, filter by country/topic/date/source, occasionally search an exact code.

- **What text gets embedded**: headline + lead, not the full body (averaging too much text blurs meaning).
- **Which model**: `all-MiniLM-L6-v2` (384 dims) + a sparse model for exact terms.
- **Chunk size**: headline + lead fits the 256-token limit here.
- **Payload fields**: whatever you filter on, here that's country, topic, source, date.

```yaml
payload:
  country: string         # indexed
  topic: string           # indexed
  source: string          # indexed
  published_at: datetime  # indexed
  headline: string        # embedded and returned
  lead: string            # embedded and returned
  body: string            # returned only, never embedded
```

Build order: create the collection, create payload indexes, then ingest. Indexes created before ingestion let HNSW add filter-aware edges as it builds.


In [ ]:
from qdrant_client import QdrantClient, models

DENSE_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
SPARSE_MODEL = "Qdrant/bm25"

client = QdrantClient(
    url="https://xyz-example.eu-west-1-0.aws.cloud.qdrant.io",
    api_key="<your-api-key>",
)

client.create_collection(
    collection_name="news",
    vectors_config={
        "dense": models.VectorParams(size=384, distance=models.Distance.COSINE),
    },
    sparse_vectors_config={
        "sparse": models.SparseVectorParams(modifier=models.Modifier.IDF),
    },
)

for field in ["country", "topic", "source"]:
    client.create_payload_index(
        collection_name="news",
        field_name=field,
        field_schema=models.PayloadSchemaType.KEYWORD,
    )

client.create_payload_index(
    collection_name="news",
    field_name="published_at",
    field_schema=models.PayloadSchemaType.DATETIME,
)

In [ ]:
ARTICLES = [
    {
        "country": "VN", "topic": "shipping", "source": "reuters",
        "published_at": "2026-07-15T08:00:00Z",
        "headline": "Port congestion worsens at Ho Chi Minh City terminals",
        "lead": "Waiting times at the city's two main berths have roughly"
                " tripled since June, and carriers are diverting boxes.",
        "body": "The backlog began with a monsoon shutdown.",
    },
    {
        "country": "VN", "topic": "shipping", "source": "nikkei",
        "published_at": "2026-07-18T08:00:00Z",
        "headline": "MAERSK-B.CO delisting rumor denied by carrier",
        "lead": "The carrier called weekend reports of a Copenhagen"
                " delisting unfounded, with no board discussion held.",
        "body": "Shares closed flat on Friday ahead of the statement.",
    },
    {
        "country": "SG", "topic": "shipping", "source": "caixin",
        "published_at": "2026-07-20T08:00:00Z",
        "headline": "Singapore berth waiting times fall for a third week",
        "lead": "Average waits at Tuas dropped below 12 hours, easing a"
                " backlog that built through the second quarter.",
        "body": "The port authority credited two new berths.",
    },
]

points = []
for i, article in enumerate(ARTICLES):
    embedded = f"{article['headline']}. {article['lead']}"
    points.append(
        models.PointStruct(
            id=i,
            vector={
                "dense": models.Document(text=embedded, model=DENSE_MODEL),
                "sparse": models.Document(text=embedded, model=SPARSE_MODEL),
            },
            payload=article,
        )
    )

client.upsert(collection_name="news", points=points)

In [ ]:
QUERY = "port congestion in Southeast Asia"

news_filter = models.Filter(
    must=[
        models.FieldCondition(key="country", match=models.MatchValue(value="VN")),
        models.FieldCondition(
            key="published_at",
            range=models.DatetimeRange(gte="2026-07-01T00:00:00Z"),
        ),
    ]
)

results = client.query_points(
    collection_name="news",
    prefetch=[
        models.Prefetch(
            query=models.Document(text=QUERY, model=DENSE_MODEL),
            using="dense", filter=news_filter, limit=50,
        ),
        models.Prefetch(
            query=models.Document(text=QUERY, model=SPARSE_MODEL),
            using="sparse", filter=news_filter, limit=50,
        ),
    ],
    query=models.RrfQuery(rrf=models.Rrf()),
    limit=10,
)

for point in results.points:
    print(f"{point.score:.4f}  {point.payload['headline']}")

# Expected output:
#   1.0000  Port congestion worsens at Ho Chi Minh City terminals
#   0.3333  MAERSK-B.CO delisting rumor denied by carrier

### Try it yourself

Embedding the full body instead of just headline + lead closes the score gap between relevant and irrelevant results (more shared vocabulary, less distinct meaning per vector).


In [ ]:
client.create_collection(
    collection_name="news_with_body",
    vectors_config={
        "dense": models.VectorParams(size=384, distance=models.Distance.COSINE),
    },
)

points = []
for i, article in enumerate(ARTICLES):
    embedded = f"{article['headline']}. {article['lead']} {article['body']}"
    points.append(
        models.PointStruct(
            id=i,
            vector={"dense": models.Document(text=embedded, model=DENSE_MODEL)},
            payload=article,
        )
    )

client.upsert(collection_name="news_with_body", points=points)

for name in ["news", "news_with_body"]:
    hits = client.query_points(
        collection_name=name,
        query=models.Document(text=QUERY, model=DENSE_MODEL),
        using="dense",
        limit=2,
    ).points
    print(f"{name}  gap {hits[0].score - hits[1].score:.4f}")
    for hit in hits:
        print(f"   {hit.score:.4f}  {hit.payload['headline']}")

## As the collection grows

- **Index vs quality**: `m` and `ef_construct` control HNSW graph accuracy vs indexing time/memory. Defaults suit most cases.
- **Memory**: quantization compresses vectors for less memory at a measurable precision cost. On-disk vectors go further, trading latency for capacity.
- **Indexing lag**: `client.get_collection(name)` reports `points_count` vs `indexed_vectors_count`. A growing gap means ingestion is outpacing indexing.

## Growing past one machine

**Sharding** splits points across nodes. **Replication** copies shards so search survives a node failure. Add nodes only after measuring and tuning the index, more nodes won't fix an unindexed filter.

## RAG (optional)

Retrieval-Augmented Generation sends your top-K results to a language model, which writes an answer. If the answer is weak, check retrieval before reaching for a bigger model.

## Where it runs

Managed Cloud, Hybrid Cloud, Private Cloud, Docker (all server modes), or Local/Edge (embedded in a process). Choose based on how much you want to operate and where the data must live.

## Design your own system

1. What do queries look like: plain language, exact codes, or both?
2. Which fields must every search filter on?
3. What is the retrieval unit: document, chunk, or image?
4. How much data, at what rate?
5. Where must the data live?

## Further reading

- [Sizing Tool](https://sizing.qdrant.tech)
- [What Is RAG](https://qdrant.tech/articles/what-is-rag-in-ai/)
- [Qdrant Cloud](https://cloud.qdrant.io/)

Next: `Module5.ipynb`, the capstone: multimodal supplier risk intelligence.
